#### Preparation

In [1]:
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [3]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"
WEIGHTS_DIR = "../weights"
BATCH_SIZE = 32
NUM_EPOCHS_FREEZE = 8
NUM_EPOCHS_FINETUNE = 20
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
weights = ResNet50_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,
                         std=std)
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,
                         std=std)
])


In [5]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [6]:
model = resnet50(weights=weights)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\yugil/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:02<00:00, 47.3MB/s]


In [7]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False


In [8]:
# Replace final layer for dataset classes
in_features = model.fc.in_features

model.fc = nn.Linear(in_features, num_classes)

In [9]:
model = model.to(DEVICE)

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=LR
)

In [11]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [12]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1 (Freeze Backbone)

In [13]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(NUM_EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")




-----------Starting Phase 1 Training-----------



Training Epoch 1/8: 100%|██████████| 30/30 [02:00<00:00,  4.02s/it]



Epoch [1/8]
Train Loss: 40.0558 | Train Acc: 0.4375
Val Loss: 10.4155 | Val Acc: 0.6125
Precision: 0.6758 | Recall: 0.6125 | F1: 0.6156


Training Epoch 2/8: 100%|██████████| 30/30 [01:55<00:00,  3.86s/it]



Epoch [2/8]
Train Loss: 36.9641 | Train Acc: 0.6312
Val Loss: 9.7679 | Val Acc: 0.7042
Precision: 0.7461 | Recall: 0.7042 | F1: 0.7010


...


Epoch [6/8]
Train Loss: 28.1525 | Train Acc: 0.8333
Val Loss: 8.0404 | Val Acc: 0.8542
Precision: 0.8587 | Recall: 0.8542 | F1: 0.8544


Training Epoch 7/8: 100%|██████████| 30/30 [01:56<00:00,  3.88s/it]



Epoch [7/8]
Train Loss: 26.2854 | Train Acc: 0.8479
Val Loss: 7.6718 | Val Acc: 0.8500
Precision: 0.8515 | Recall: 0.8500 | F1: 0.8497


Training Epoch 8/8: 100%|██████████| 30/30 [01:57<00:00,  3.92s/it]



Epoch [8/8]
Train Loss: 24.8962 | Train Acc: 0.8615
Val Loss: 7.3711 | Val Acc: 0.8500
Precision: 0.8528 | Recall: 0.8500 | F1: 0.8493


#### Training 2 Fine-Tune

In [14]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [15]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [16]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(NUM_EPOCHS_FINETUNE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, NUM_EPOCHS_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ResNet50.pth"))


-----------Starting Fine-tuning Stage-----------



Training Epoch 1/20: 100%|██████████| 30/30 [01:58<00:00,  3.94s/it]



Epoch [1/20]
Train Loss: 23.0453 | Train Acc: 0.8667
Val Loss: 6.4794 | Val Acc: 0.8750
Precision: 0.8770 | Recall: 0.8750 | F1: 0.8750


Training Epoch 2/20: 100%|██████████| 30/30 [01:57<00:00,  3.90s/it]



Epoch [2/20]
Train Loss: 20.0707 | Train Acc: 0.8781
Val Loss: 5.7688 | Val Acc: 0.8792
Precision: 0.8810 | Recall: 0.8792 | F1: 0.8790


...


Epoch [18/20]
Train Loss: 5.1385 | Train Acc: 0.9542
Val Loss: 1.9083 | Val Acc: 0.9375
Precision: 0.9394 | Recall: 0.9375 | F1: 0.9377


Training Epoch 19/20: 100%|██████████| 30/30 [01:55<00:00,  3.85s/it]



Epoch [19/20]
Train Loss: 4.6707 | Train Acc: 0.9542
Val Loss: 1.8729 | Val Acc: 0.9292
Precision: 0.9299 | Recall: 0.9292 | F1: 0.9291


Training Epoch 20/20: 100%|██████████| 30/30 [01:54<00:00,  3.82s/it]



Epoch [20/20]
Train Loss: 4.1581 | Train Acc: 0.9604
Val Loss: 1.8138 | Val Acc: 0.9375
Precision: 0.9407 | Recall: 0.9375 | F1: 0.9378


#### Predicting Sample

In [17]:
from PIL import Image

checkpoint = torch.load("../weights/ResNet50.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


print(predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf blight\leaf blight_val_21.jpg"))

C:\Users\yugil\AppData\Local\Temp\ipykernel_2356\4138783925.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../weights/ResNet50.pth")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\e6va6je238\\Desktop\\Cassava_Leaf_Datasets\\Classification\\val\\leaf blight\\leaf blight_val_21.jpg'